# Lab 2 — Sparse vs dense vs hybrid (EP10)

จาก Lab 1: คำถามยาวที่มีชื่อตำรา ทำให้ชิ้น "library card" ตกไปอันดับ 8 และคำตอบที่อยู่ในคลังก็หายไปด้วย
แล็บนี้เราลองวิธีค้นอื่นตาม §16.3 บนคลังเดิม

**สิ่งที่จะได้ทำ**
1. สร้างคลังเดิมจาก Lab 1 (เขียนไว้ให้แล้ว)
2. เขียน **BM25** เอง ตามสมการ 16.4
3. ดูว่าเมื่อไร BM25 ชนะ และเมื่อไร dense ชนะ
4. เขียน **RRF** (สมการ 16.8) ตรวจกับตัวเลขในตำรา แล้วรวมสองแบบเป็น **hybrid**
5. วัดผลด้วยคำถาม 10 ข้อที่รู้เฉลยว่าอยู่หน้าไหน: **hit@3**

แล็บนี้ไม่เรียก Gemma เลย ใช้แค่ embedding model

> อ้างอิง: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §16.3 — CC BY-SA 4.0

## 0. เตรียมเครื่องก่อนเริ่ม

เหมือน Lab 1: Python 3.10+ หรือ Colab และ `OPENROUTER_API_KEY` (Colab: 🔑 Secrets · เครื่องตัวเอง: ไฟล์ `.env`)
แล็บนี้ใช้แค่ embedding model — ค่าใช้จ่ายราว 0.05 เซ็นต์ดอลลาร์
วิธีตั้งค่า key และเหตุผลที่เลือก model อยู่ใน Lab 1

In [ ]:
%pip install -q "openai>=1.50" python-dotenv "pypdf>=6" numpy "tokenizers==0.23.3"

## 1. คลังเดิมจาก Lab 1

โค้ดใน cell ถัดไปคือ Lab 1 ส่วนที่ 3–5 ย่อรวมกัน: โหลดตำรา v1 หน้า 292–374 → ตัดชิ้น 512/64 → embedding

In [ ]:
import os, re, math, urllib.request
from collections import Counter
import numpy as np
from openai import OpenAI
from pypdf import PdfReader
from tokenizers import Tokenizer

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

EMBED_MODEL = "qwen/qwen3-embedding-8b"
EMBED_EXTRA = {"provider": {"order": ["nebius"], "allow_fallbacks": False}}
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key(), max_retries=5)
tok = Tokenizer.from_pretrained("google/gemma-4-26b-a4b-it")

urllib.request.urlretrieve("https://arxiv.org/pdf/2606.24937v1", "hitchhikers-guide-v1.pdf")
reader = PdfReader("hitchhikers-guide-v1.pdf")
HEADER = "H. Roitman — The Hitchhiker’s Guide to Agentic AI: From Foundations to Systems\n"
pages = [{"source": "hitchhikers-guide-v1.pdf", "page": n,
          "text": reader.pages[n - 1].extract_text().replace(HEADER, "")}
         for n in range(292, 375)]

def chunk_page(page, W=512, O=64):
    ids = tok.encode(page["text"], add_special_tokens=False).ids
    return [{"source": page["source"], "page": page["page"],
             "text": tok.decode(ids[start:start + W])}
            for start in range(0, max(len(ids) - O, 1), W - O)]

def embed(texts):
    vectors = []
    for i in range(0, len(texts), 64):
        r = client.embeddings.create(model=EMBED_MODEL, input=texts[i:i + 64], extra_body=EMBED_EXTRA)
        vectors += [d.embedding for d in r.data]
    V = np.array(vectors)
    return V / np.linalg.norm(V, axis=1, keepdims=True)

chunks = [c for p in pages for c in chunk_page(p)]
E = embed([c["text"] for c in chunks])
print(len(chunks), "chunks,", E.shape)

In [ ]:
def dense_rank(query):
    scores = E @ embed([query])[0]
    return list(np.argsort(-scores))

## 2. BM25 ขั้นที่ 1: สถิติของคลัง

$$\text{BM25}(d,q)=\sum_{t\in q}\text{IDF}(t)\cdot\frac{f(t,d)\,(k_1+1)}{f(t,d)+k_1\left(1-b+b\,\frac{|d|}{\text{avgdl}}\right)}\qquad(16.4)$$

ก่อนให้คะแนน ต้องรู้: จำนวนชิ้น `N` · ความยาวเฉลี่ย `avgdl` · แต่ละคำอยู่ในกี่ชิ้น `df`
ตำราเขียน `IDF(t)` ไว้โดยไม่ได้ให้สูตร — เราใช้รูปที่ BM25 ใช้กันทั่วไป: $\ln\left(\frac{N-n+0.5}{n+0.5}+1\right)$ เมื่อ `n` = จำนวนชิ้นที่มีคำนั้น

In [ ]:
def words(text):
    return re.findall(r"\w+", text.lower())

docs = [words(c["text"]) for c in chunks]
N = len(docs)
avgdl = sum(len(d) for d in docs) / N
df = Counter(w for d in docs for w in set(d))
print(N, "chunks · avgdl =", round(avgdl), "words")
print("df['rag'] =", df["rag"], "· df['plaid'] =", df["plaid"])

In [ ]:
def idf(t):
    n = df.get(t, 0)
    return math.log((N - n + 0.5) / (n + 0.5) + 1)
print("idf('rag') =", round(idf("rag"), 2), "· idf('plaid') =", round(idf("plaid"), 2))

## 3. BM25 ขั้นที่ 2: ให้คะแนนทุกชิ้น

สามความคิดจากคลิปทฤษฎี: คำหายากนับมาก (`IDF`) · คำที่โผล่หลายครั้งได้เพิ่มแต่ค่อยๆ อิ่มตัว (`k1`) · ชิ้นยาวถูกหักเล็กน้อย (`b`)
ตำรา: $k_1 \in [1.2, 2.0]$, $b = 0.75$ — เราใช้ $k_1 = 1.5$

In [ ]:
def bm25_rank(query, k1=1.5, b=0.75):
    terms = words(query)
    scores = []
    for d in docs:
        tf = Counter(d)
        s = 0.0
        for t in terms:
            f = tf[t]
            s += idf(t) * f * (k1 + 1) / (f + k1 * (1 - b + b * len(d) / avgdl))
        scores.append(s)
    return list(np.argsort(-np.array(scores)))

def show(name, ranked, k=3):
    print(f"{name:7s}", [chunks[i]["page"] for i in ranked[:k]])

## 4. ใครชนะเมื่อไร

ตำรา: sparse ยังชนะเรื่อง **คำที่ต้องตรงตัว** (รหัส ชื่อเฉพาะ คำหายาก) · dense ชนะเรื่อง **ความหมาย** เมื่อคำถามใช้คำต่างจากเอกสาร

In [ ]:
q = "What is PLAID?"
show("bm25", bm25_rank(q))
show("dense", dense_rank(q))

In [ ]:
q = "Which chunks should I give the model if small ones retrieve better but lack context?"
show("bm25", bm25_rank(q))
show("dense", dense_rank(q))

## 5. Hybrid ด้วย RRF

$$\text{RRF}(d)=\sum_{r\in R}\frac{1}{k+\text{rank}_r(d)},\quad k=60\qquad(16.8)$$

ใช้ **อันดับ** แทนคะแนน เพราะคะแนน BM25 กับ cosine คนละสเกล — ตัวอย่างในตำรา: อันดับ 3 กับ 7 → 0.0308 · อันดับ 1 ทั้งคู่ → 0.0328
ฟังก์ชันนี้คือ `reciprocal_rank_fusion` ใน Listing 16.8 ของตำรา

In [ ]:
def rrf(ranked_lists, k=60):
    scores = {}
    for ranked in ranked_lists:
        for rank, doc in enumerate(ranked, start=1):
            scores[doc] = scores.get(doc, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)

bm = ["x", "y", "d", "z"]                       # d at rank 3
de = ["p", "q", "r", "s", "t", "u", "d"]        # d at rank 7
print(round(1 / (60 + 3) + 1 / (60 + 7), 4), round(2 / 61, 4))
print(rrf([bm, de])[:3])

In [ ]:
def hybrid_rank(query):
    return rrf([bm25_rank(query), dense_rank(query)])

q = "When should I fine-tune a model instead of looking things up?"
for name, fn in [("bm25", bm25_rank), ("dense", dense_rank), ("hybrid", hybrid_rank)]:
    show(name, fn(q))

## 6. วัดผล: hit@3 บนคำถาม 10 ข้อ

แต่ละข้อรู้เฉลยว่าคำตอบอยู่หน้าไหน (`gold`) · `first_hit` = อันดับแรกที่ได้ชิ้นจากหน้าเฉลย · **hit@3** = เจอในสามอันดับแรกหรือไม่
มีทั้งคำถามแบบคำหายาก (PLAID, HyDE, FAISS) และแบบถามด้วยคำของเราเอง

In [ ]:
TESTS = [
    ("What is PLAID?", {301}),
    ("What is HyDE?", {303}),
    ("What does FAISS stand for?", {298}),
    ("How can I search millions of vectors quickly?", {298}),
    ("Which method compares every query token with every document token?", {300}),
    ("How do I make keyword search understand synonyms?", {299}),
    ("When should I fine-tune a model instead of looking things up?", {296}),
    ("Which chunks should I give the model if small ones retrieve better but lack context?", {302, 303}),
    ("How do I combine keyword search and embedding search?", {298}),
    ("What analogy does the textbook use to explain RAG?", {295}),
]

def first_hit(ranked, gold):
    for rank, i in enumerate(ranked, start=1):
        if chunks[i]["page"] in gold:
            return rank

print(f"{'question':58s} bm25 dense hybrid")
hits = Counter()
for q, gold in TESTS:
    row = []
    for name, fn in [("bm25", bm25_rank), ("dense", dense_rank), ("hybrid", hybrid_rank)]:
        r = first_hit(fn(q), gold)
        hits[name] += r <= 3
        row.append(r)
    print(f"{q[:56]:58s} {row[0]:4d} {row[1]:5d} {row[2]:6d}")
print("hit@3:", dict(hits), "of", len(TESTS))

## 7. คำถามที่พลาดใน Lab 1

คำถามยาวที่มีชื่อตำรา — hybrid ช่วยได้ไหม

In [ ]:
question = ("In Roitman's book The Hitchhiker's Guide to Agentic AI, "
            "what analogy does the RAG chapter use to explain RAG?")
for name, fn in [("bm25", bm25_rank), ("dense", dense_rank), ("hybrid", hybrid_rank)]:
    print(name, first_hit(fn(question), {295}))

## สรุป

| | ที่เราเห็นใน lab |
|---|---|
| IDF | `rag` อยู่ใน 34 ชิ้น → ≈ 1.4 · `plaid` อยู่ใน 1 ชิ้น → ≈ 4.5 |
| BM25 ชนะ | "What is PLAID?" → หน้า 301 อันดับ 1 · dense หาไม่เจอ |
| Dense ชนะ | ถามเรื่อง parent-child โดยไม่ใช้คำนั้น → หน้า 302–303 อันดับต้น · BM25 หาไม่เจอ |
| RRF | อันดับ 3 + 7 → 0.0308 · 1 + 1 → 0.0328 ตรงกับตำรา |
| Hybrid | fine-tune vs look up: ไม่ติดสามอันดับแรกในรายการไหน แต่ hybrid ได้อันดับ 1 |
| hit@3 (10 ข้อ) | BM25 4 · dense 4 · **hybrid 8** — แต่ PLAID กับคำถามภาพเปรียบแย่ลง |
| คำถามที่พลาดใน Lab 1 | BM25 10 · dense 8 · hybrid 5 — ยังพลาด |

**แบบฝึกหัด**
1. ลอง `k1 = 1.2` และ `k1 = 2.0` (ช่วงในตำรา) — hit@3 ของ BM25 เปลี่ยนไหม
2. ลอง `rrf(…, k=1)` แทน 60 — อันดับต้นๆ ได้เปรียบมากขึ้น hit@3 ของ hybrid เป็นอย่างไร
3. เขียนคำถามของคุณเองอีก 5 ข้อพร้อมหน้าเฉลย แล้ววัดใหม่

**ถัดไป — Lab 3:** ตัดคลังเดียวกันหลายแบบ แล้ววัด hit@3 แบบเดียวกัน